# 02 模型评测

**目标**：调用模型 API，收集每条样本的回答，输出 `results/raw/raw_results.csv`

**建议顺序**：
1. 先跑 DeepSeek（便宜，验证流程）
2. 流程通后再加 Kimi / Qwen-Long

⚠️ **前置条件**：已在 `.env` 文件中配置 API Key

In [19]:
import sys
sys.path.insert(0, '..')

import os
from pathlib import Path
import pandas as pd
import yaml
from dotenv import load_dotenv

load_dotenv('../.env')

from src.eval_runner import run_eval, call_model, get_client

config = yaml.safe_load(open('../configs/eval_config.yaml', encoding='utf-8'))
print('配置加载成功')

配置加载成功


## Step 1：检查 API Key 配置

In [20]:
key_map = {
    'deepseek': 'DEEPSEEK_API_KEY',
    'kimi':     'MOONSHOT_API_KEY',
    'qwen':     'DASHSCOPE_API_KEY',
}
print('API Key 状态：')
for model, env_var in key_map.items():
    val = os.getenv(env_var, '')
    status = '✅ 已配置' if val and not val.startswith('sk-xxx') else '❌ 未配置'
    masked = (val[:8] + '...' + val[-4:]) if len(val) > 12 else '（空）'
    print(f'  {model:12s} ({env_var}): {status}  {masked}')

API Key 状态：
  deepseek     (DEEPSEEK_API_KEY): ✅ 已配置  sk-5cfdc...a922
  kimi         (MOONSHOT_API_KEY): ✅ 已配置  sk-sExnE...ifpB
  qwen         (DASHSCOPE_API_KEY): ✅ 已配置  sk-a1bce...10e4


## Step 2：单条冒烟测试（验证 API 连通性）

In [21]:
TEST_MODEL = 'deepseek'  # 改为你想测试的模型

try:
    client, model_name = get_client(TEST_MODEL, config)
    test_context = '本公司于2024年1月1日在北京成立，注册资本为1000万元人民币，法人代表为张三。'
    test_question = '公司的注册资本是多少？'
    
    response, tokens, latency = call_model(
        client, model_name,
        context=test_context,
        question=test_question,
        max_tokens=64,
    )
    print(f'✅ [{TEST_MODEL}] 连通性测试通过')
    print(f'   问题: {test_question}')
    print(f'   回答: {response}')
    print(f'   Tokens: {tokens}  Latency: {latency:.2f}s')
except Exception as e:
    print(f'❌ 连通性测试失败: {e}')

✅ [deepseek] 连通性测试通过
   问题: 公司的注册资本是多少？
   回答: 1000万元人民币
   Tokens: 96  Latency: 0.63s


## Step 3：批量评测

调整下方 `MODEL_KEYS` 和 `MAX_SAMPLES` 控制评测范围：

In [22]:
# ── 可调参数 ──────────────────────────────────
MODEL_KEYS = ['deepseek', 'kimi', 'qwen']       # 要评测的模型，可选: 'deepseek', 'kimi', 'qwen'
MAX_SAMPLES = None              # None = 跑全量（105 条/模型，共 315 次 API 调用）
RESUME = True                   # True = 断点续跑，跳过已有结果
# ────────────────────────────────────────────

dataset_path = '../data/processed/niah_dataset.jsonl'

if not Path(dataset_path).exists():
    print('⚠️  数据集不存在，请先运行 01_data_preparation.ipynb')
else:
    df = run_eval(
        dataset_path=dataset_path,
        model_keys=MODEL_KEYS,
        config=config,
        output_dir='../results/raw',
        max_samples=MAX_SAMPLES,
        resume=RESUME,
    )
    print(f'\n完成！共 {len(df)} 条结果')
    df.head(5)

📂 加载 105 条样本，来自: ../data/processed/niah_dataset.jsonl
   已有 60 条结果，启用断点续跑

🚀 [deepseek] deepseek-chat — RPM 限制: 60


deepseek:  31%|███▏      | 33/105 [00:36<01:01,  1.17req/s]

  ⚠️  API 调用失败（尝试 1/3）: Error code: 400 - {'error': {'message': 'Content Exists Risk', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_request_error'}}
  ⚠️  API 调用失败（尝试 2/3）: Error code: 400 - {'error': {'message': 'Content Exists Risk', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_request_error'}}
  ⚠️  API 调用失败（尝试 3/3）: Error code: 400 - {'error': {'message': 'Content Exists Risk', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_request_error'}}


deepseek:  70%|███████   | 74/105 [01:42<00:56,  1.83s/req]

  ⚠️  API 调用失败（尝试 1/3）: Error code: 400 - {'error': {'message': 'Content Exists Risk', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_request_error'}}
  ⚠️  API 调用失败（尝试 2/3）: Error code: 400 - {'error': {'message': 'Content Exists Risk', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_request_error'}}
  ⚠️  API 调用失败（尝试 3/3）: Error code: 400 - {'error': {'message': 'Content Exists Risk', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_request_error'}}


...

  ⚠️  API 调用失败（尝试 1/3）: Error code: 400 - {'error': {'message': 'Input data may contain inappropriate content. For details, see: https://help.aliyun.com/zh/model-studio/error-code#inappropriate-content', 'type': 'data_inspection_failed', 'param': None, 'code': 'data_inspection_failed'}, 'id': 'chatcmpl-e702f4a0-4048-9126-b51a-1dc52bff6893', 'request_id': 'e702f4a0-4048-9126-b51a-1dc52bff6893'}
  ⚠️  API 调用失败（尝试 2/3）: Error code: 400 - {'error': {'message': 'Input data may contain inappropriate content. For details, see: https://help.aliyun.com/zh/model-studio/error-code#inappropriate-content', 'type': 'data_inspection_failed', 'param': None, 'code': 'data_inspection_failed'}, 'id': 'chatcmpl-5d93539b-6473-9fe4-bf70-2ea0b939b2f6', 'request_id': '5d93539b-6473-9fe4-bf70-2ea0b939b2f6'}
  ⚠️  API 调用失败（尝试 3/3）: Error code: 400 - {'error': {'message': 'Input data may contain inappropriate content. For details, see: https://help.aliyun.com/zh/model-studio/error-code#inappropriate-content', 't

qwen:  96%|█████████▌| 101/105 [03:30<00:19,  4.85s/req]

  ⚠️  API 调用失败（尝试 1/3）: Error code: 400 - {'error': {'message': 'Input data may contain inappropriate content. For details, see: https://help.aliyun.com/zh/model-studio/error-code#inappropriate-content', 'type': 'data_inspection_failed', 'param': None, 'code': 'data_inspection_failed'}, 'id': 'chatcmpl-8ad904ea-c045-9a28-b662-d8e82488b5da', 'request_id': '8ad904ea-c045-9a28-b662-d8e82488b5da'}
  ⚠️  API 调用失败（尝试 2/3）: Error code: 400 - {'error': {'message': 'Input data may contain inappropriate content. For details, see: https://help.aliyun.com/zh/model-studio/error-code#inappropriate-content', 'type': 'data_inspection_failed', 'param': None, 'code': 'data_inspection_failed'}, 'id': 'chatcmpl-a516c2d3-bff1-96ef-bbc0-53e88837e3c0', 'request_id': 'a516c2d3-bff1-96ef-bbc0-53e88837e3c0'}
  ⚠️  API 调用失败（尝试 3/3）: Error code: 400 - {'error': {'message': 'Input data may contain inappropriate content. For details, see: https://help.aliyun.com/zh/model-studio/error-code#inappropriate-content', 't

qwen: 100%|██████████| 105/105 [03:50<00:00,  2.20s/req]


✅ 结果保存至: ../results/raw/raw_results.csv（共 282 条）

完成！共 282 条结果


## Step 4：快速预览结果

In [23]:
results_path = '../results/raw/raw_results.csv'

if Path(results_path).exists():
    df = pd.read_csv(results_path)
    print(f'结果统计：{len(df)} 条')
    print(f'\n模型分布：')
    print(df['model'].value_counts().to_string())
    print(f'\n深度分布：')
    print(df['depth_pct'].value_counts().sort_index().to_string())
    print(f'\n前 5 条样本：')
    display(df[['model', 'context_length', 'depth_pct', 'expected_answer', 'model_response', 'latency_s']].head(5))

结果统计：282 条

模型分布：
model
deepseek    94
kimi        94
qwen        94

深度分布：
depth_pct
0      39
10     39
25     39
50     39
75     48
90     33
100    45

前 5 条样本：


,model,context_length,depth_pct,expected_answer,model_response,latency_s
0,deepseek,2000,0,47.3亿元人民币,47.3亿元人民币,0.78
1,deepseek,2000,10,EMP-2024-08831,EMP-2024-08831,0.42
2,deepseek,2000,25,暗星计划，2024年11月8日,内部项目代号为“暗星计划”，正式启动日期为2024年11月8日。,0.70
3,deepseek,2000,50,47.3亿元人民币,47.3亿元人民币,0.49
4,deepseek,2000,75,47.3亿元人民币,47.3亿元人民币。,0.45


## Step 5：估算 Token 消耗 & 费用

In [ ]:
from src.metrics import calc_cost

if Path(results_path).exists():
    df = pd.read_csv(results_path)
    cost_df = calc_cost(df)
    print('Token 消耗 & 真实费用（区分 input/output/cache）：\n')
    print(cost_df.to_string(index=False))
    print(f'\n💰 总费用: ¥{cost_df["cost_cny"].sum():.4f}')

Token 消耗 & 费用估算：
  deepseek    :    498,951 tokens  ≈ ¥0.499
  kimi        :    569,309 tokens  ≈ ¥6.832
  qwen        :    521,904 tokens  ≈ ¥2.088


## ✅ 评测完成

下一步：打开 `03_analysis_visualization.ipynb` 进行数据分析和可视化